## Importation des librairies utiles

In [45]:
pip install ultralytics transformers numpy matplotlib torch torchvision scikit-learn seaborn tqdm

Note: you may need to restart the kernel to use updated packages.


In [46]:
import json
import os
import numpy as np
import yaml

### Conversion du dataset dans le format attendu par le modèle

In [47]:
chemin_annotations = os.path.join("dataset", "annotations.json")
dossier_labels = os.path.join("dataset", "labels")
os.makedirs(dossier_labels, exist_ok=True)

with open(chemin_annotations, "r", encoding="utf-8") as fichier:
    donnees = json.load(fichier)

coco_map_seen = {
    "Bicycle": 1,
    "Dog": 16,
    "Pizza": 53,
    "Bus": 5,
    "Chair": 56
}

annotations_par_image = {}
for annot in donnees.get("annotations", []):
    image_id = annot["image_id"]
    if image_id not in annotations_par_image:
        annotations_par_image[image_id] = []
    annotations_par_image[image_id].append(annot)
            
coco_map = {"Bicycle": 1, "Dog": 16, "Pizza": 53, "Bus": 5, "Chair": 56}
coco_map_unseen = {
    "Guitar": 5,      
    "Microphone": 6,     
    "Sunglasses": 7,
    "Helmet": 8,
    "Drum": 9
}


## YOLO

### Chargement du modèle

In [48]:
from ultralytics import YOLO
modele_yolo = YOLO("yolo11n.pt")

### Fonctions pour la classification

In [49]:
chemin_annotations = os.path.join("dataset", "annotations.json")
dossier_images = os.path.join("dataset", "images")
dossier_labels = os.path.join("dataset", "labels")
os.makedirs(dossier_labels, exist_ok=True)

with open(chemin_annotations, "r", encoding="utf-8") as fichier:
    donnees = json.load(fichier)

map_classes = {}
id_classe = 0
for categorie in donnees.get("classes", {}).get("seen", []):
    map_classes[categorie] = id_classe
    id_classe += 1
for categorie in donnees.get("classes", {}).get("unseen", []):
    map_classes[categorie] = id_classe
    id_classe += 1

images_info = {
    img["id"]: {
        "file_name": img["file_name"], 
        "width": img["width"], 
        "height": img["height"]
    } for img in donnees.get("images", [])
}

annotations_par_image = {}
for annot in donnees.get("annotations", []):
    image_id = annot["image_id"]
    if image_id not in annotations_par_image:
        annotations_par_image[image_id] = []
    annotations_par_image[image_id].append(annot)

for image_id, annots in annotations_par_image.items():
    if image_id not in images_info:
        continue
        
    info = images_info[image_id]
    img_w, img_h = info["width"], info["height"]
    
    nom_base = os.path.splitext(info["file_name"])[0]
    chemin_txt = os.path.join(dossier_labels, f"{nom_base}.txt")
    
    with open(chemin_txt, "w", encoding="utf-8") as f:
        for annot in annots:
            cat = annot["category"]
            if cat in coco_map_seen:
                classe_id = coco_map_seen[cat]
                x, y, w, h = annot["bbox"] 
                
                x_center = (x + (w / 2)) / img_w
                y_center = (y + (h / 2)) / img_h
                w_norm = w / img_w
                h_norm = h / img_h

                f.write(f"{classe_id} {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}\n")

noms_classes = [k for k, v in sorted(map_classes.items(), key=lambda item: item[1])]
yaml_path = os.path.join("dataset", "data.yaml")

contenu_yaml = {
    "path": os.path.abspath("dataset"), 
    "train": "images",                  
    "val": "images",                    
    "nc": len(noms_classes),
    "names": noms_classes
}

with open(yaml_path, "w") as f:
    yaml.dump(contenu_yaml, f, sort_keys=False)
    
print(f"data.yaml généré dans : {yaml_path}")
with open("dataset/annotations.json", "r") as f:
    data = json.load(f)

def test_classification(type_test="seen"):
    tp, fp, fn = 0, 0, 0
    
    categories_cibles = data.get("classes", {}).get(type_test, [])
    
    coco_ids = {
        "Bicycle": 1, "Bus": 5, "Dog": 16, "Pizza": 53, "Chair": 56,
        "Guitar": 1001, "Microphone": 1002, "Sunglasses": 1003, "Helmet": 1004, "Drum": 1005
    }
    
    vraies_classes_par_image = {}
    for annot in data.get("annotations", []):
        cat = annot["category"]
        if cat in categories_cibles:
            img_id = annot["image_id"]
            if img_id not in vraies_classes_par_image:
                vraies_classes_par_image[img_id] = set()
            vraies_classes_par_image[img_id].add(coco_ids.get(cat, -1))
            
    print(f"Évaluation des classes '{type_test}' sur {len(vraies_classes_par_image)} images...")
    
    for image_id, vraies_classes in vraies_classes_par_image.items():
        img_info = next((img for img in data.get("images", []) if img["id"] == image_id), None)
        
        if img_info:
            img_path = os.path.join("dataset", "images", img_info["file_name"])
            
            resultats = modele_yolo(img_path, verbose=False)
            
            pred_classes = set()
            for res in resultats:
                for box in res.boxes:
                    pred_classes.add(int(box.cls[0].item()))
        
            tp += len(vraies_classes.intersection(pred_classes))
            
            fn += len(vraies_classes.difference(pred_classes))
            
            ids_cibles_possibles = set([coco_ids.get(c, -1) for c in categories_cibles])
            predictions_cibles = pred_classes.intersection(ids_cibles_possibles)
            
            fp += len(predictions_cibles.difference(vraies_classes))
                    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    rappel = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1_score = 2 * (precision * rappel) / (precision + rappel) if (precision + rappel) > 0 else 0
    
    return {"precision": precision, "rappel": rappel, "f1_score": f1_score}

data.yaml généré dans : dataset/data.yaml


### Test sur les classes vues

In [50]:
resultats_vus = test_classification("seen")
print("Résultats pour les classes vues :", resultats_vus)

Évaluation des classes 'seen' sur 129 images...
Résultats pour les classes vues : {'precision': 0.9663865546218487, 'rappel': 0.8914728682170543, 'f1_score': 0.9274193548387097}


In [51]:
def test_classification_par_classe(type_test="seen"):
    categories_cibles = data.get("classes", {}).get(type_test, [])
    
    coco_ids = {
        "Bicycle": 1, "Bus": 5, "Dog": 16, "Pizza": 53, "Chair": 56,
        "Guitar": 1001, "Microphone": 1002, "Sunglasses": 1003, "Helmet": 1004, "Drum": 1005
    }
    
    tp = {cat: 0 for cat in categories_cibles}
    fp = {cat: 0 for cat in categories_cibles}
    fn = {cat: 0 for cat in categories_cibles}
    
    vraies_classes_par_image = {}
    for annot in data.get("annotations", []):
        cat = annot["category"]
        if cat in categories_cibles:
            img_id = annot["image_id"]
            if img_id not in vraies_classes_par_image:
                vraies_classes_par_image[img_id] = set()
            vraies_classes_par_image[img_id].add(coco_ids.get(cat, -1))
            
    print(f"Évaluation détaillée des classes '{type_test}' sur {len(vraies_classes_par_image)} images...\n")
    
    for image_id, vraies_classes_ids in vraies_classes_par_image.items():
        img_info = next((img for img in data.get("images", []) if img["id"] == image_id), None)
        
        if img_info:
            img_path = os.path.join("dataset", "images", img_info["file_name"])
            resultats = modele_yolo(img_path, verbose=False)
            
            pred_classes_ids = set()
            for res in resultats:
                for box in res.boxes:
                    pred_classes_ids.add(int(box.cls[0].item()))
            
            for cat in categories_cibles:
                cat_id = coco_ids.get(cat, -1)
                
                est_present = cat_id in vraies_classes_ids
                est_predit = cat_id in pred_classes_ids
                
                if est_present and est_predit:
                    tp[cat] += 1
                elif not est_present and est_predit:
                    fp[cat] += 1
                elif est_present and not est_predit:
                    fn[cat] += 1
                    
    resultats_finaux = {}
    print(f"{'Classe':<12} | {'Précision':<10} | {'Rappel':<10} | {'F1-Score':<10} | {'Support (Nb images)'}")
    print("-" * 65)
    
    for cat in categories_cibles:
        p = tp[cat] / (tp[cat] + fp[cat]) if (tp[cat] + fp[cat]) > 0 else 0
        r = tp[cat] / (tp[cat] + fn[cat]) if (tp[cat] + fn[cat]) > 0 else 0
        f1 = 2 * (p * r) / (p + r) if (p + r) > 0 else 0
        support = tp[cat] + fn[cat] 
        
        resultats_finaux[cat] = {"precision": p, "rappel": r, "f1_score": f1}
        
        print(f"{cat:<12} | {p:<10.2f} | {r:<10.2f} | {f1:<10.2f} | {support}")
        
    return resultats_finaux

In [52]:
resultats_par_classe = test_classification_par_classe()
print(resultats_par_classe)

Évaluation détaillée des classes 'seen' sur 129 images...

Classe       | Précision  | Rappel     | F1-Score   | Support (Nb images)
-----------------------------------------------------------------
Bicycle      | 0.94       | 0.77       | 0.85       | 22
Dog          | 0.96       | 1.00       | 0.98       | 23
Pizza        | 1.00       | 0.97       | 0.98       | 31
Bus          | 0.97       | 0.94       | 0.95       | 32
Chair        | 0.94       | 0.71       | 0.81       | 21
{'Bicycle': {'precision': 0.9444444444444444, 'rappel': 0.7727272727272727, 'f1_score': 0.85}, 'Dog': {'precision': 0.9583333333333334, 'rappel': 1.0, 'f1_score': 0.9787234042553191}, 'Pizza': {'precision': 1.0, 'rappel': 0.967741935483871, 'f1_score': 0.9836065573770492}, 'Bus': {'precision': 0.967741935483871, 'rappel': 0.9375, 'f1_score': 0.9523809523809523}, 'Chair': {'precision': 0.9375, 'rappel': 0.7142857142857143, 'f1_score': 0.8108108108108109}}


### Test sur les classes non vues

In [53]:
resultats_non_vus = test_classification("unseen")
print("Résultats pour les classes non vues :", resultats_non_vus)

Évaluation des classes 'unseen' sur 122 images...
Résultats pour les classes non vues : {'precision': 0, 'rappel': 0.0, 'f1_score': 0}
